# 01 — Data Collection
This notebook fetches historical daily weather data for Abeokuta, Ogun State (lat=7.16, lon=3.35) using the Open-Meteo Historical Weather API, aggregates hourly data to daily, and saves it to `data/raw/ogun_weather.csv`.

## Why Open-Meteo?
- Free and reliable historical weather datasets (ERA5 backfill).
- Global coverage including Abeokuta, Nigeria.
- Simple REST API with rich variables (temperature, humidity, wind, precipitation, pressure).

In [1]:
# Setup: ensure project root on sys.path for 'src' imports
import sys
import os

# Add parent directory (which contains src) to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
print('Updated sys.path:', sys.path)

Updated sys.path: ['/usr/local/python/3.12.1/lib/python312.zip', '/usr/local/python/3.12.1/lib/python3.12', '/usr/local/python/3.12.1/lib/python3.12/lib-dynload', '', '/home/codespace/.local/lib/python3.12/site-packages', '/usr/local/python/3.12.1/lib/python3.12/site-packages', '/workspaces/climate-ml-lab']


In [2]:
from src.data_loader import fetch_hourly, aggregate_daily, save_daily_csv
from src.utils import RAW_DIR
import pandas as pd

# Fetch and save data (handles defaults: from 2015-01-01 to today)
try:
    hourly_df = fetch_hourly()
    daily_df = aggregate_daily(hourly_df)
    out_path = save_daily_csv(daily_df)
    print('Saved to:', out_path)
except Exception as e:
    print('Fetch failed, trying to load existing CSV if present. Error:', e)
    csv_path = RAW_DIR / 'ogun_weather.csv'
    if csv_path.exists():
        daily_df = pd.read_csv(csv_path)
        print('Loaded existing file:', csv_path)
    else:
        raise

daily_df.head()

2025-12-30 08:48:35 | INFO | src.data_loader | Requesting hourly data from Open-Meteo: {'latitude': 7.16, 'longitude': 3.35, 'start_date': '2015-01-01', 'end_date': '2025-12-30', 'hourly': 'temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,surface_pressure', 'timezone': 'Africa/Lagos'}
2025-12-30 08:48:38 | INFO | src.data_loader | Fetched 96408 hourly rows spanning 2015-01-01 01:00:00+01:00 to 2025-12-31 00:00:00+01:00
2025-12-30 08:48:39 | INFO | src.data_loader | Saved daily data to /workspaces/climate-ml-lab/data/raw/ogun_weather.csv (4018 rows)


Saved to: /workspaces/climate-ml-lab/data/raw/ogun_weather.csv


,date,temperature_c,humidity_pct,wind_speed_mps,precipitation_mm,surface_pressure_hpa
0,2015-01-01 00:00:00+01:00,28.013043,53.521739,5.821739,0.0,10.001043
1,2015-01-02 00:00:00+01:00,27.583333,66.708333,5.304167,0.0,10.008292
2,2015-01-03 00:00:00+01:00,28.266667,62.791667,5.458333,0.0,10.021000
3,2015-01-04 00:00:00+01:00,27.716667,44.041667,6.500000,0.0,10.026333
4,2015-01-05 00:00:00+01:00,26.291667,28.291667,6.079167,0.0,10.018958


In [3]:
daily_df.tail()

,date,temperature_c,humidity_pct,wind_speed_mps,precipitation_mm,surface_pressure_hpa
4013,2025-12-27 00:00:00+01:00,28.670833,75.041667,9.000000,0.6,9.996458
4014,2025-12-28 00:00:00+01:00,29.037500,71.875000,7.970833,0.2,9.997500
4015,2025-12-29 00:00:00+01:00,28.645833,73.958333,8.712500,0.4,10.009708
4016,2025-12-30 00:00:00+01:00,28.266667,77.875000,6.570833,6.5,10.014458
4017,2025-12-31 00:00:00+01:00,26.200000,87.000000,10.500000,0.0,10.025000


In [4]:
# Basic shape and dtypes
daily_df.shape, daily_df.dtypes

((4018, 6),
 date                    datetime64[ns, Africa/Lagos]
 temperature_c                                float64
 humidity_pct                                 float64
 wind_speed_mps                               float64
 precipitation_mm                             float64
 surface_pressure_hpa                         float64
 dtype: object)

In [5]:
# Generate processed dataset if missing
from src.preprocess import load_raw, preprocess, save_processed
from src.utils import PROCESSED_DIR

csv_out = PROCESSED_DIR / 'clean_weather.csv'
try:
    proc_df = preprocess(load_raw())
    save_processed(proc_df)
    print('Processed data saved to:', csv_out)
except Exception as e:
    print('Preprocess failed:', e)

2025-12-30 08:49:31 | INFO | src.preprocess | Saved processed data to /workspaces/climate-ml-lab/data/processed/clean_weather.csv (4017 rows, 13 cols)


Processed data saved to: /workspaces/climate-ml-lab/data/processed/clean_weather.csv
